# Dog


In [11]:
import torch
torch.set_float32_matmul_precision("high")
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import matplotlib.pyplot as plt
import shapiq 
import src

In [12]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
# model = CLIPModel.from_pretrained("google/siglip2-base-patch32-256")
# processor = CLIPProcessor.from_pretrained("google/siglip2-base-patch32-256")

In [13]:
input_text = "giraffe"
input_image = Image.open("assets/giraffe_drinking.jpg")

In [14]:
game = src.game_huggingface.VisionLanguageGame(
    model=model,
    processor=processor,
    input_image=input_image,
    input_text=input_text,
    batch_size=64
)
n_players_image = game.n_players_image
n_players_text = game.n_players_text

In [15]:
fixlip = src.fixlip.FIxLIP(
    n_players_image=n_players_image,
    n_players_text=n_players_text,
    max_order=2,
    p=0.5,  # weight
    mode="banzhaf",
    random_state=0,
    approximation_type="regression",
)
fixlip_proxyshap_noadjustment = src.fixlip.FIxLIP(
    n_players_image=n_players_image,
    n_players_text=n_players_text,
    max_order=2,
    p=0.5,  # weight
    mode="banzhaf",
    random_state=0,
    approximation_type="proxyshap-noadjustment",
)

In [16]:
import time

src.utils.set_seed(0)
BUDGET = 2048
a = time.time()
interaction_values = fixlip.approximate(game, budget=BUDGET)
b = time.time()
print("FIxLIP runtime (s): ", b - a)
# interaction_values_proxyshap = fixlip_proxyshap.approximate(game, budget=BUDGET)
# c = time.time()
# print("FIxLIP ProxySHAP runtime (s): ", c - b)
# interaction_values_proxyspex = fixlip_proxyspex.approximate(game, budget=BUDGET)
d = time.time()
# print("FIxLIP ProxySpex runtime (s): ", d - c)
interaction_values_no_adjustment = fixlip_proxyshap_noadjustment.approximate(game, budget=BUDGET)
e = time.time()
print("FIxLIP No Adjustment runtime (s): ", e - d)

/Users/santothies/Desktop/msr_int_iq/fixlip_experiments/src/fixlip.py:331: UserWarning: Index `FWBII` is not a valid interaction index. Valid indices are: SII, BII, CHII, Co-Moebius, SGV, BGV, CHGV, IGV, EGV, k-SII, STII, FSII, kADD-SHAP, FBII, SV, BV, JointSV, Moebius, ELC.
  pad_interactionvalues = shapiq.InteractionValues(
/Users/santothies/Desktop/msr_int_iq/shapiq_local/src/shapiq/interaction_values.py:526: UserWarning: Index `FWBII` is not a valid interaction index. Valid indices are: SII, BII, CHII, Co-Moebius, SGV, BGV, CHGV, IGV, EGV, k-SII, STII, FSII, kADD-SHAP, FBII, SV, BV, JointSV, Moebius, ELC.
  return InteractionValues(


FIxLIP runtime (s):  61.87380003929138
FIxLIP No Adjustment runtime (s):  41.93764615058899


In [17]:
def denormalize(img, mean, std):
    return img * torch.tensor(std).view(3, 1, 1) + torch.tensor(mean).view(3, 1, 1)

In [18]:
text_tokens = game.inputs.tokens()
text_tokens = text_tokens[0:game.n_players_text]
text_tokens = [token.replace('▁', '') for token in text_tokens]
assert len(text_tokens) == game.n_players_text
players_text = list(range(game.n_players_image, game.n_players))
assert game.n_players == interaction_values_no_adjustment.n_players == max(players_text) + 1
input_image_processed = game.inputs['pixel_values'].squeeze(0)
input_image_denormalized = src.utils.denormalize(
    input_image_processed, 
    game.processor.image_processor.image_mean, 
    game.processor.image_processor.image_std
).permute(1, 2, 0).numpy()


AttributeError: 

# FIxLIP Baseline

In [ ]:
src.plot.plot_image_and_text_together(
    img=input_image_denormalized,
    text=text_tokens,
    image_players=list(range(n_players_image)),
    iv=interaction_values,
    plot_interactions=True,
    top_k=16,
    normalize_jointly=True,
    figsize=(6.5, 6.5),
    fontsize=22,
    margin=0.3,
    color_text=True,
    plot_heatmap=True,
    show=False,
    max_value=3
)
plt.tight_layout(pad=0.15)

NameError: name 'input_image_denormalized' is not defined

# ProxySHAP

In [ ]:
src.plot.plot_image_and_text_together(
    img=input_image_denormalized,
    text=text_tokens,
    image_players=list(range(n_players_image)),
    iv=interaction_values_no_adjustment,
    plot_interactions=True,
    top_k=16,
    normalize_jointly=True,
    figsize=(6.5, 6.5),
    fontsize=22,
    margin=0.3,
    color_text=True,
    plot_heatmap=True,
    show=False,
    max_value=3
)
plt.tight_layout(pad=0.15)